# Lesson 4: Edge cases, errors and bugs

This notebook goes with [Lesson 4 of *Unit Testing in Python*](https://acatlin.github.io/python-unit-testing/lessons/0004-edge-cases-errors-and-bugs.html). The explanations are on that page. The code is here, ready to run.

- Run the cells from top to bottom, one at a time: click a cell and press Shift+Enter.
- This is your own copy. Change anything you like; nothing you do here alters the course. To keep your work, choose File, then Save a copy in Drive.

## Set up

Colab comes with older versions of pandas and pytest than the lesson page was written with. The next cell installs the versions the page uses, so that what you see here matches it. It can take a minute.

pip may end with a message that other packages in Colab want the older pandas. This notebook does not use them, so carry on. If Colab asks you to restart the session, do that, then continue from the cell after this one.

In [ ]:
%pip install -q "pandas==3.1.1" "pytest==9.2.0"

Check what you have. A line that starts with `!` goes to the computer's command line, not to Python: it is this notebook's stand-in for the terminal.

In [ ]:
!python --version
!python -m pytest --version
!python -c "import pandas; print('pandas', pandas.__version__)"

## How files and commands work here

The lesson has you save files in an editor and type commands in a terminal. A notebook has neither, so two stand-ins do the job.

- **`%%writefile name`** on the first line of a cell saves the rest of the cell as a file, in place of running it. To change the file, edit the cell and run it again. (You can also open the file from the folder icon at the left edge of Colab.)
- **`!`** at the start of a line sends it to the command line. Every `!` line starts in the notebook's own folder, so the commands below begin with `cd cleaning-project &&` to step into the project first.

**What will look different from the lesson page.** The page shows output captured on Windows with Python 3.14. Colab runs Linux with its own Python. So the `platform` line says `linux` and another Python version, `rootdir:` is a folder under `/content`, paths are written with `/` in place of `\`, the rules of `=` signs may be wider or narrower, and the timings differ. The results are the same: the dots and `F`s, the lines marked `E`, and the counts on the last line.

## The project so far

The project as Lesson 3 left it: seven tests, all passing.

In [ ]:
import os

os.makedirs("cleaning-project/tests", exist_ok=True)

In [ ]:
%%writefile cleaning-project/clean.py
"""Data-cleaning helpers: the starting point for Lesson 2.

Each function is small on purpose: small functions are easy to test.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01, summarise_by


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]


def test_parse_money_reads_thousands_without_cents():
    assert parse_money(pd.Series(["$3,000"])).tolist() == [3000]


def test_summarise_by_counts_rows_and_averages_within_each_group():
    staff = pd.DataFrame({
        "dept": ["a", "a", "b"],
        "pay": [10, 20, 60],
    })
    expected = pd.DataFrame({
        "dept": ["a", "b"],
        "n": [2, 1],
        "mean": [15.0, 60.0],
    })
    pd.testing.assert_frame_equal(summarise_by(staff, "dept", "pay"), expected)


def test_summarise_by_returns_a_data_frame_with_the_documented_columns():
    staff = pd.DataFrame({"dept": ["a", "a", "b"], "pay": [10, 20, 60]})
    out = summarise_by(staff, "dept", "pay")
    assert isinstance(out, pd.DataFrame)
    assert list(out.columns) == ["dept", "n", "mean"]


def test_rescale01_returns_one_float_per_input_value():
    x = rescale01(pd.Series([1, 2, 3]))
    assert x.dtype == "float64"
    assert len(x) == 3
    assert x.max() <= 1

In [ ]:
!cd cleaning-project && python -m pytest

## Ask what could go wrong

Missing values first. The decision: they stay missing, and the rest are rescaled as usual.

The next cell starts with `%%writefile -a`. The `-a` means append: the cell is added to the end of the test file, the way you would type a new test under the others. **Run each `-a` cell once only.** Run one twice and the test is in the file twice. If you get in a muddle, go back to the last cell that saves the whole test file and run the cells from there in order.

In [ ]:
%%writefile -a cleaning-project/tests/test_clean.py


def test_rescale01_keeps_missing_values_missing():
    result = rescale01(pd.Series([1, None, 3]))
    expected = pd.Series([0, None, 1])
    pd.testing.assert_series_equal(result, expected)

It passes already (`8 passed`). That is a fine outcome: the behaviour is now written down.

In [ ]:
!cd cleaning-project && python -m pytest

## A bug, caught test-first

What does `rescale01()` do with a column where every value is the same? Ask Python, from the project folder.

In [ ]:
!cd cleaning-project && python -c "import pandas as pd; from clean import rescale01; print(rescale01(pd.Series([5, 5, 5])))"

Zero divided by zero: `NaN`, with no error and no warning. That is a bug.

### Step 1: write the test and watch it fail

All zeros is a sensible answer. Say so, and run. **This run is meant to fail**: `1 failed, 8 passed`.

In [ ]:
%%writefile -a cleaning-project/tests/test_clean.py


def test_rescale01_returns_zeros_when_every_value_is_the_same():
    assert rescale01(pd.Series([5, 5, 5])).tolist() == [0, 0, 0]

In [ ]:
!cd cleaning-project && python -m pytest

### Step 2: fix the function

Two new lines in `rescale01()`.

In [ ]:
%%writefile cleaning-project/clean.py
"""Data-cleaning helpers: the starting point for Lesson 2.

Each function is small on purpose: small functions are easy to test.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    lo, hi = x.min(), x.max()
    if lo == hi:
        return x * 0.0  # every value is the same: all zeros, missing kept
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

### Step 3: run everything again

The new test passes, and so does every old one: `9 passed`.

In [ ]:
!cd cleaning-project && python -m pytest

## Expecting an error

For the wrong type, the right behaviour is to stop with a clear message. The test names the type of error and a piece of its message. `pytest.raises()` belongs to pytest, so the file needs `import pytest` at the top; `import warnings` goes in at the same time, for the section after this one.

Imports go at the top of a file, where `-a` cannot reach, so this cell saves the whole test file again: the nine tests so far, the new imports, and the new test at the end.

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import warnings

import pandas as pd
import pytest

from clean import clean_names, parse_money, rescale01, summarise_by


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]


def test_parse_money_reads_thousands_without_cents():
    assert parse_money(pd.Series(["$3,000"])).tolist() == [3000]


def test_summarise_by_counts_rows_and_averages_within_each_group():
    staff = pd.DataFrame({
        "dept": ["a", "a", "b"],
        "pay": [10, 20, 60],
    })
    expected = pd.DataFrame({
        "dept": ["a", "b"],
        "n": [2, 1],
        "mean": [15.0, 60.0],
    })
    pd.testing.assert_frame_equal(summarise_by(staff, "dept", "pay"), expected)


def test_summarise_by_returns_a_data_frame_with_the_documented_columns():
    staff = pd.DataFrame({"dept": ["a", "a", "b"], "pay": [10, 20, 60]})
    out = summarise_by(staff, "dept", "pay")
    assert isinstance(out, pd.DataFrame)
    assert list(out.columns) == ["dept", "n", "mean"]


def test_rescale01_returns_one_float_per_input_value():
    x = rescale01(pd.Series([1, 2, 3]))
    assert x.dtype == "float64"
    assert len(x) == 3
    assert x.max() <= 1


def test_rescale01_keeps_missing_values_missing():
    result = rescale01(pd.Series([1, None, 3]))
    expected = pd.Series([0, None, 1])
    pd.testing.assert_series_equal(result, expected)


def test_rescale01_returns_zeros_when_every_value_is_the_same():
    assert rescale01(pd.Series([5, 5, 5])).tolist() == [0, 0, 0]


def test_rescale01_rejects_a_column_that_is_not_numeric():
    with pytest.raises(TypeError, match="must be a numeric column"):
        rescale01(pd.Series(["a", "b"]))

**This run is meant to fail** (`1 failed, 9 passed`). The error is of the right type, but its message is a cryptic one from deep inside the arithmetic.

In [ ]:
!cd cleaning-project && python -m pytest

The test is red, so improve the function: check the input at the top.

In [ ]:
%%writefile cleaning-project/clean.py
"""Data-cleaning helpers: the starting point for Lesson 2.

Each function is small on purpose: small functions are easy to test.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    if not pd.api.types.is_numeric_dtype(x):
        raise TypeError("x must be a numeric column")
    lo, hi = x.min(), x.max()
    if lo == hi:
        return x * 0.0  # every value is the same: all zeros, missing kept
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

In [ ]:
!cd cleaning-project && python -m pytest

## Warnings

Two more tests: one says that junk input warns, the other that well-formed input is quiet. After them you have the twelve tests of the finished project.

In [ ]:
%%writefile -a cleaning-project/tests/test_clean.py


def test_parse_money_warns_when_a_value_cannot_be_parsed():
    with pytest.warns(UserWarning, match="could not be parsed"):
        parse_money(pd.Series(["n/a"]))


def test_parse_money_is_quiet_when_every_value_parses():
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        parse_money(pd.Series(["$5"]))

In [ ]:
!cd cleaning-project && python -m pytest

A warning that no test asked for does not fail the test, but it is not hidden either. Here is a scratch file with one test that calls `parse_money(pd.Series(["n/a"]))` outside `pytest.warns()`. Look for the `warnings summary` and the count on the last line: `13 passed, 1 warning`.

In [ ]:
%%writefile cleaning-project/tests/test_demo.py
import pandas as pd

from clean import parse_money


def test_parse_money_turns_junk_into_missing():
    assert parse_money(pd.Series(["n/a"])).isna().all()

In [ ]:
!cd cleaning-project && python -m pytest

In [ ]:
os.remove("cleaning-project/tests/test_demo.py")

## Try it

Step 1 of the lesson is what you have just done.

2. What should `rescale01(pd.Series([7]))`, a single value, return? Decide, write the test in a `%%writefile -a` cell, and run it.

In [ ]:
# step 2: your test

In [ ]:
!cd cleaning-project && python -m pytest

3. What does `clean_names()` do with `[""]`, with `["123 Go!"]`, or with a bare string such as `"First Name"` in place of a list? Try them in the cell below, decide whether you are happy with each result, and write a test that records your decision.

In [ ]:
!cd cleaning-project && python -c "from clean import clean_names; print(clean_names(['']))"

In [ ]:
# step 3: your test

In [ ]:
!cd cleaning-project && python -m pytest

Now go back to the [lesson page](https://acatlin.github.io/python-unit-testing/lessons/0004-edge-cases-errors-and-bugs.html) for the quiz.